# SSL 및 API 연결 테스트 

# API -> 데이터프레임 -> csv 저장

In [17]:
import os
import requests
import pandas as pd
import time
from tqdm import tqdm
from datetime import datetime, timedelta
import ssl
import warnings
from requests.packages.urllib3.exceptions import InsecureRequestWarning
from dotenv import load_dotenv
load_dotenv()

True

In [36]:

# SSL 및 경고 설정
warnings.filterwarnings('ignore', category=InsecureRequestWarning)
ssl._create_default_https_context = ssl._create_unverified_context

# API 기본 설정
API_KEY = os.getenv('DO_API_KEY')
BASE_URL = 'https://apis.data.go.kr/B552845/katSale/trades'

# 도매시장 코드 불러오기
df_market = pd.read_csv('도매시장_코드.csv', encoding='cp949')

# 품목 코드 설정
ITEM_CODES = {
    "양파": "1201",
    "배추": "1001",
    "상추": "1005",
    "사과": "0601"
}

# 날짜 입력 (yyyy-MM-dd 형식)
start_date = '2018-01-03'
end_date = '2018-01-31'

# 날짜 처리
start_dt = datetime.strptime(start_date, '%Y-%m-%d')
end_dt = datetime.strptime(end_date, '%Y-%m-%d')
total_days = (end_dt - start_dt).days + 1

# 설정
max_retries = 3
FAIL_LOG = []

# 품목별 반복
for item_name, code in tqdm(ITEM_CODES.items(), desc="전체 품목 진행"):
    LARGE = code[:2]
    MID = code[2:]
    data_list = []

    print(f"\n📦 {item_name} 수집 시작: {start_date} ~ {end_date}")
    print("📈 진행률: ", end='')

    current_dt = start_dt
    count = 0

    while current_dt <= end_dt:
        date_str = current_dt.strftime('%Y-%m-%d')  # API 포맷 그대로

        for mcode, market_name in df_market.values:
            retry_count = 0
            market_success = False

            while retry_count < max_retries:
                params = {
                    'serviceKey': API_KEY,
                    'pageNo': 1,
                    'numOfRows': 100,
                    'cond[trd_clcln_ymd::EQ]': date_str,
                    'cond[whsl_mrkt_cd::EQ]': mcode,
                    'cond[gds_lclsf_cd::EQ]': LARGE,
                    'cond[gds_mclsf_cd::EQ]': MID
                }

                try:
                    response = requests.get(BASE_URL, params=params, verify=False, timeout=10)
                    if response.status_code == 200:
                        json_data = response.json()
                        items = json_data.get('response', {}).get('body', {}).get('items', {}).get('item', [])
                        if isinstance(items, list) and items:
                            data_list.extend(items)
                            market_success = True
                            time.sleep(0.2)
                        break
                    else:
                        retry_count += 1
                except Exception:
                    retry_count += 1
                    time.sleep(0.5)

            if not market_success:
                fail_reason = ""
                try:
                    if response.status_code != 200:
                        fail_reason = f"HTTP {response.status_code}: {response.text[:100]}"
                    else:
                        # 200인데도 데이터 없음
                        result_msg = json_data.get('response', {}).get('header', {}).get('resultMsg', 'No data')
                        fail_reason = f"200 OK but no data: {result_msg}"
                except Exception as e:
                    fail_reason = f"Exception while parsing failure: {str(e)}"

                FAIL_LOG.append({
                    "item": item_name,
                    "market": market_name,
                    "mcode": mcode,
                    "date": date_str,
                    "reason": fail_reason
                })

        # 진척도 출력
        count += 1
        if count % 10 == 0:
            percent = int((count / total_days) * 100)
            print(f"{percent}%", end='', flush=True)
        else:
            print('.', end='', flush=True)

        current_dt += timedelta(days=1)
        time.sleep(0.2)

    print(f"\n✅ {item_name} 완료: {len(data_list):,}건")

    if data_list:
        df = pd.DataFrame(data_list)
        filename = f"data/유통공사_도매시장_{item_name}_{start_date.replace('-', '')}-{end_date.replace('-', '')}.csv"
        df.to_csv(filename, encoding='cp949', index=False)
        print(f"💾 저장됨: {filename}")
    else:
        print(f"⚠️ {item_name}: 수집된 데이터 없음")

# 실패 로그 저장ㅁ
if FAIL_LOG:
    df_fail = pd.DataFrame(FAIL_LOG)
    df_fail.to_csv('data/유통공사_fail_log.csv', index=False, encoding='cp949')
    print(f"\n❗ 실패 요청 {len(FAIL_LOG)}건 기록됨: data/유통공사_fail_log.csv")
else:
    print("\n🎉 모든 수집 성공, 실패 없음!")


전체 품목 진행:   0%|                                                                            | 0/4 [00:00<?, ?it/s]


📦 양파 수집 시작: 2018-01-03 ~ 2018-01-31
📈 진행률: .........34%.........68%.........
✅ 양파 완료: 8,122건


전체 품목 진행:  25%|████████████████▊                                                  | 1/4 [06:52<20:37, 412.52s/it]

💾 저장됨: data/유통공사_도매시장_양파_20180103-20180131.csv

📦 배추 수집 시작: 2018-01-03 ~ 2018-01-31
📈 진행률: .........34%.........68%.........
✅ 배추 완료: 10,557건


전체 품목 진행:  50%|█████████████████████████████████▌                                 | 2/4 [16:31<17:00, 510.47s/it]

💾 저장됨: data/유통공사_도매시장_배추_20180103-20180131.csv

📦 상추 수집 시작: 2018-01-03 ~ 2018-01-31
📈 진행률: .........34%.........68%.........
✅ 상추 완료: 19,444건


전체 품목 진행:  75%|██████████████████████████████████████████████████▎                | 3/4 [23:40<07:53, 473.39s/it]

💾 저장됨: data/유통공사_도매시장_상추_20180103-20180131.csv

📦 사과 수집 시작: 2018-01-03 ~ 2018-01-31
📈 진행률: .........34%.........68%.........
✅ 사과 완료: 38,949건


전체 품목 진행: 100%|███████████████████████████████████████████████████████████████████| 4/4 [29:47<00:00, 446.75s/it]

💾 저장됨: data/유통공사_도매시장_사과_20180103-20180131.csv

❗ 실패 요청 763건 기록됨: data/유통공사_fail_log.csv
